# Simple HandWriting OCR

This simple handwriting will involve custom CNN model by using Tensorflow.
<br>I will call it MinimNet and it can be found in model.py

## 1. Import All and Prepare the Dataset
Before going on, we could download the dataset from the [link Sueiras](https://s3-eu-west-1.amazonaws.com/handwriting-curated-database/curated.tar.gz) have given.<br>
And just unzip the file in the same directory of this notebook.

In [1]:
from model2 import MinimNet
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from imutils import resize, grab_contours
from matplotlib import pyplot as plt
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelBinarizer
from PIL import Image
import numpy as np
import cv2
import os
import bitstring

I0000 00:00:1782638251.395001    7328 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


import all needed libraries and functions needed

And before we are going to prepare the dataset, we would want to set our parameters first

1. Dataset path,
2. Image size,
3. Batch size,
4. Number of epochs,
5. Initial Learning Rate (LR)
6. Ratio of train/test

In [2]:
notebook_path = os.path.abspath('Simple-OCR_runpod.ipynb')
dataset_path = os.path.dirname(notebook_path)+'/curated'
etl1_dir = os.path.join(os.path.dirname(notebook_path), 'ETL/ETL1')
etl1_paths = [
    os.path.join(etl1_dir, f) for f in os.listdir(etl1_dir) if f.startswith('ETL1C_')
] if os.path.exists(etl1_dir) else []

image_size = (32, 32)
batch_size = 128
epoch = 50
init_lr = 1e-1
test_ratio = .25

After putting all the needed parameters, we then will try to load the images from the path we defined

We will need both the image and label, which the label is the folder name (ASCII Decimal)

Thus we need somekind of label to make it more uniformed.<br> Then we will make a classLabels that consist of needed images to train.

After that we will load all of the images that we need along with the labels<br>
And change the labels into a one-hot label.

In [3]:
classLabels = "1234567890ABCDEFGHIJKLMNOPQRSTUVWXYZ"
classLabels = [i for i in classLabels]

img = []
lbl = []

# Duyệt qua các thư mục trong dataset
for folder in os.listdir(dataset_path):
    folder_path = os.path.join(dataset_path, folder)
    
    # Kiểm tra xem có phải là thư mục không
    if os.path.isdir(folder_path):
        # Tránh lỗi nếu tên thư mục không phải là số (VD: thư mục 'logs', 'models')
        try:
            folder_char = chr(int(folder))
        except ValueError:
            continue
            
        # Nếu ký tự thư mục nằm trong danh sách nhãn cần lấy
        if folder_char in classLabels:
            lbl_idx = classLabels.index(folder_char)
            
            for file in os.listdir(folder_path):
                # BẢO VỆ 1: Bỏ qua các file ẩn của macOS / Linux (.DS_Store, ._filename.png)
                if file.startswith('.'):
                    continue
                
                file_path = os.path.join(folder_path, file)
                
                # BẢO VỆ 2: Khối Try-Except chặn đứng mọi lỗi đọc ảnh hỏng
                try:
                    # Mở ảnh, resize, chuẩn hóa và thêm chiều channel
                    image_obj = Image.open(file_path).resize(image_size)
                    img_array = np.array(image_obj) * (1. / 255.)
                    img_tensor = np.expand_dims(img_array, axis=-1)
                    
                    img.append(img_tensor)
                    lbl.append(lbl_idx)
                except Exception as e:
                    # Nếu gặp file không phải ảnh, in ra cảnh báo và tiếp tục vòng lặp
                    print(f"⚠️ Bỏ qua file lỗi: {file_path} | Chi tiết: {e}")

# Chuyển đổi sang Numpy Array
img = np.array(img, dtype='float32') # Ép kiểu float32 rõ ràng để tối ưu bộ nhớ
lbl = np.array(lbl)

print(f"✅ Đã tải thành công {len(img)} ảnh từ curated dataset.")

# Đọc thêm dữ liệu từ ETL1
def load_etl1_data(file_paths, class_labels, img_size=(32, 32)):
    etl_img = []
    etl_lbl = []
    
    for file_path in file_paths:
        if not os.path.exists(file_path):
            print(f"⚠️ ETL file không tồn tại: {file_path}")
            continue
            
        print(f"Đang đọc ETL file: {file_path}...")
        file_size = os.path.getsize(file_path)
        record_size = 2052
        num_records = file_size // record_size
        
        with open(file_path, "rb") as f:
            for i in range(num_records):
                _bytes = f.read(record_size)
                if len(_bytes) < record_size:
                    break
                    
                # Parse M-Type record
                f_bit = bitstring.ConstBitStream(bytes=_bytes)
                r = f_bit.readlist('uint:16,bytes:2,uint:16,hex:8,hex:8,4*uint:8,uint:32,4*uint:16,4*uint:8,pad:32,bytes:2016,pad:32')
                char_code = r[1][0]
                img_bytes = r[18]
                
                # Giải mã ký tự
                if 32 <= char_code <= 126:
                    char = chr(char_code)
                    if char in class_labels:
                        # Giải mã 4-bit gray levels (0-15) thành 8-bit (0-255) và đảo màu (255 - val)
                        pixels = []
                        for b in img_bytes:
                            p1 = (b >> 4) & 0x0F
                            p2 = b & 0x0F
                            pixels.extend([p1, p2])
                            
                        img_array = 255 - (np.array(pixels, dtype=np.uint8) * 17)
                        img_array = img_array.reshape(63, 64)
                        
                        # Convert sang PIL, resize và chuẩn hóa
                        img_obj = Image.fromarray(img_array).resize(img_size)
                        img_tensor = np.expand_dims(np.array(img_obj) * (1. / 255.), axis=-1)
                        
                        etl_img.append(img_tensor)
                        etl_lbl.append(class_labels.index(char))
                        
    return etl_img, etl_lbl

if len(etl1_paths) > 0:
    etl_img, etl_lbl = load_etl1_data(etl1_paths, classLabels, image_size)
    if len(etl_img) > 0:
        img = np.concatenate([img, np.array(etl_img, dtype='float32')], axis=0)
        lbl = np.concatenate([lbl, np.array(etl_lbl)], axis=0)
        print(f"✅ Đã tải thêm {len(etl_img)} ảnh từ ETL1.")

print(f"✅ Tổng số ảnh sau khi gộp: {len(img)}")
print(f"Shape của tập dữ liệu: {img.shape}")

✅ Đã tải thành công 26562 ảnh.
Shape của tập dữ liệu: (26562, 32, 32, 1)


In [4]:
#create a one-hot label
le = LabelBinarizer()
lbl = le.fit_transform(lbl)

#count the total number of images per class
classTotals = lbl.sum(axis=0)
classWeight = {}

#create a custom weight for every class
for i in range(len(classTotals)):
    classWeight[i] = classTotals.max() / classTotals[i]
    
#split the class to train and test using scikit train_test_split
(trainImg, testImg, trainLbl, testLbl) = train_test_split(img, lbl,
                                                         test_size = test_ratio,
                                                         stratify = lbl,
                                                         random_state = 42)

After we done in splitting the dataset, we will proceed to the next step.
<br><br>

## 2. Create the Model to train

As I have said before, we will try to build a model.<br>
This model will be consisting of some layers such as:
1. Bottleneck Layer,
2. Inverted Bottleneck Layer,
3. Squeeze Global Layer,
4. SE Module.

The model could be checked in the Model.py or in the summary below.

In [5]:
model = MinimNet((image_size[0], image_size[1], 1), len(classTotals))
model.summary()

from tensorflow.keras.optimizers.schedules import InverseTimeDecay
lr_schedule = InverseTimeDecay(
    initial_learning_rate=init_lr,
    decay_steps=1,
    decay_rate=init_lr / epoch
)
opt = SGD(learning_rate=lr_schedule)
model.compile(loss="categorical_crossentropy", optimizer=opt, metrics=['accuracy'])

I0000 00:00:1782638256.811313    7328 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 10115 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3080 Ti, pci bus id: 0000:2e:00.0, compute capability: 8.6


Model: "MinimNet_MaxAcc"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 32, 32, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 32, 32,    │        288 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 32, 32,    │        128 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv2d    │ (None, 32, 32,    │        288 │ activation[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        128 │ depthwise_conv2d… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 32)        │          0 │ activation_1[0][… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 1, 1, 32)  │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 1, 1, 8)   │        264 │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1, 1, 32)  │        288 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 32, 32,    │          0 │ activation_1[0][… │
│                     │ 32)               │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 32, 32,    │        512 │ multiply[0][0]    │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │         64 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 32, 32,    │      1,024 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv2d_1  │ (None, 16, 16,    │        576 │ activation_2[0][

 Total params: 2,114,748 (8.07 MB)

 Trainable params: 2,097,692 (8.00 MB)

 Non-trainable params: 17,056 (66.62 KB)

Now after we have the model, it is time to start training the model.

## 3. Train the model

Now, we will make an augmentation for the training batch so that it could have more variations of the dataset.<br>
We will also make an additional module from tensorflow such as:

1. EarlyStopping : To stop the training if there's no improvement on the monitored variable,
2. ModelCheckpoint : To save the model/weight if it reached higher value on the monitored variable,
3. ReduceLROnPlateau : To reduce the LR if there's no improvement on the monitored variable.

In [6]:
aug = ImageDataGenerator(rotation_range=10, zoom_range=.05,
                        width_shift_range=.1, height_shift_range=.1,
                        shear_range=.15, horizontal_flip=False,
                        fill_mode='nearest')

eS = EarlyStopping(monitor='val_loss', patience=10, verbose=0, mode='min')
mChk = ModelCheckpoint('OCR Model.h5', save_best_only=True, monitor='val_loss', mode='min')
rLR = ReduceLROnPlateau(monitor='val_loss', factor=.01, patience=10, verbose=1, min_lr=1e-6, mode='min')

In [ ]:
h = model.fit(aug.flow(trainImg, trainLbl, batch_size=batch_size),
             validation_data=(testImg, testLbl),
             steps_per_epoch=len(trainImg)//batch_size,
             epochs=epoch, callbacks=[eS, mChk, rLR],
             class_weight=classWeight, verbose=2)

Epoch 1/50


I0000 00:00:1782638257.927142    7328 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1782638261.750216    7549 service.cc:153] XLA service 0x7c0c6004ed40 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1782638261.750234    7549 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 3080 Ti, Compute Capability 8.6 (Driver: 13.0.0; Runtime: 12.4.0; Toolkit: 12.5.0; DNN: 9.23.2)
I0000 00:00:1782638261.950974    7549 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1782638262.635460    7549 cuda_dnn.cc:461] Loaded cuDNN version 92302
I0000 00:00:1782638262.814710    7549 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_13535__.325
I0000 00:00:1782638262.998428    7549 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hint

After training, we will plot the result of the training, such as the loss and accuracy.

Here, we will use the pyplot to plot the result into an image called Result.png

In [ ]:
n = np.array(range(len(h.history['loss'])))
plt.style.use('ggplot')
plt.figure()
plt.plot(n, h.history['loss'], label='Train_loss')
plt.plot(n, h.history['val_loss'], label='Val_loss')
plt.title=('Training Accuracy and Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss/Accuracy')
plt.legend(loc='lower left')
plt.savefig('Result.png')

Then we will test it, and show each class result in F1 score using sklearn classification_report

In [ ]:
predictions = model.predict(testImg, batch_size=batch_size)
print(classification_report(testLbl.argmax(axis=1),
                            predictions.argmax(axis=1), target_names=classLabels))

And its finished! But only for the model part.

Next, we will combine this with OpenCV to make it into real OCR and read multiple character in the images at once.

## 4. Combining with OpenCV
In this step we will:
1. Load an image containing handwriting using OpenCV,
2. Convert the image into Grayscale,
3. Apply some image processing,
4. Get the contours and sort them from left to right,
5. Crop images based on the contours,
6. Predict each image from the cropped image,
7. Append the result to the image.

In [ ]:
# put all the needed variable/parameter
file_name = 'image_to_test.jpg'
image_path = os.path.dirname(notebook_path)+'/'+file_name

image_to_test = cv2.imread(image_path)

#convert to grayscale
gray = cv2.cvtColor(image_to_test, cv2.COLOR_BGR2GRAY)
#blurring the image
blur = cv2.GaussianBlur(gray, (5, 5), 0)

#apply edge filter to the blurred image
edge = cv2.Canny(blur, 30, 150)

In [ ]:
#get contours
cntrs = cv2.findContours(edge.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
cntrs = grab_contours(cntrs)

#sort contours from top left to bottom right
boundingBox = [cv2.boundingRect(c) for c in cntrs]
(cntrs, _) = zip(*sorted(zip(cntrs, boundingBox), key=lambda x: (-x[1][1], x[1][0])))

In [ ]:
char_detected = []

for c in cntrs:
    (x,y,w,h) = cv2.boundingRect(c)
    
    #attemp to ignore small contours
    if (w >= 5 and w <= 350) and (h >= 15 and h <= 320):
        #create an Region of Interest
        roi = gray[y:y+h, x:x+w]
        
        #threshold it to make it into binary image
        bin_img = cv2.threshold(roi, 0 ,255, cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU)[1]
        (iH, iW) = bin_img.shape
        
        #if imageWidth(iW) is bigger than the Height, then we will scale the image until
        #the width is equal to 32, else scale the image until we got the height is the 32
        #32 used in here is based on the image_size which is (32,32)
        if iW > iH:
            bin_img = resize(bin_img, width = image_size[0])
        else:
            bin_img = resize(bin_img, height = image_size[1])
            
        #update with the new image size
        (iH, iW) = bin_img.shape
        dX = int(max(0, 32 - iW) / 2.0)
        dY = int(max(0, 32 - iH) / 2.0)
        
        #pad the image and resize it to match the input of the model
        padded = cv2.copyMakeBorder(bin_img, top=dY, bottom=dY,
            left=dX, right=dX, borderType=cv2.BORDER_CONSTANT,
            value=(0, 0, 0))
        padded = cv2.resize(padded, image_size)
        
        #normalize the image so that it got the same value of testImg
        padded = padded.astype("float32") / 255.0
        padded = np.expand_dims(padded, axis=-1)
        
        #put it into list of the queue of images to predict
        char_detected.append([padded, (x, y, w, h)])
        
#get each location and image of detected char
boxes = [b[1] for b in char_detected]
char_detected = np.array([c[0] for c in char_detected], dtype="float32")
#feed it as the input of the model
predictions = model.predict(char_detected)
preds = []
for pred in predictions:
    i = np.argmax(pred)
    prob = pred[i]
    preds.append([classLabels[i], prob])

for (pred, (x, y, w, h)) in zip(preds, boxes):
    #put the result into the image and print it
    print("[DETECTED CHAR] {} - {:.2f}%".format(pred[0], pred[1] * 100))
    cv2.rectangle(image_to_test, (x, y), (x + w, y + h), (0, 255, 0), 2)
    cv2.putText(image_to_test, "{}".format(pred[0]), (x - 10, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 2)

#show the result
cv2.imwrite("Image_after_test.jpg", image_to_test)

In [ ]:
#show the result
plt.imshow(image_to_test)

As we can see the result shows:
<br>
"This is only a jimple ocr 2020 thij nekds momd troaining"

There are some errors on the S and The insides of R, which always shows up as D or O.<br>
And RE that connected also read as M.

And that's it! Congratulation, you have reached the end of this tutorial!

But as we can see that the result is still not that good.<br>
There are still things that we can do to fix this such as:
1. Make the the minimum size bigger,
2. Maybe train it more or try to use other network,
3. Try to train using other dataset.